# MAAS Multi-Turn GRPO Training Notebook

This Colab notebook runs the stricter multi-turn 3-day GRPO training flow for MAAS.

Target setup:
- `meta-llama/Llama-3.2-1B-Instruct`
- Unsloth
- 4-bit loading
- multi-turn conversation-prefix training examples
- comparison plot against the single-step baseline if you upload its summary JSON

Use a `T4 GPU` runtime before running the cells.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/sparsh1258/MAAS.git

## 2. Enter the repo

In [ ]:
%cd /content/MAAS

## 3. Install dependencies

In [ ]:
!pip install -r requirements.txt
!pip install -U trl datasets transformers unsloth

If Colab asks for a restart after install, restart the runtime and rerun from the repo cell.

In [ ]:
%cd /content/MAAS

## 4. Log into Hugging Face

Paste a fresh read token in place of the placeholder before running.

In [ ]:
from huggingface_hub import login
login("PASTE_YOUR_HF_READ_TOKEN_HERE")

## 5. Optional: upload your single-step baseline summary

Upload `grpo_training_summary.json` here if you want the comparison plot to overlay the single-step baseline.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 6. Move the uploaded baseline summary into `results/`

Skip this cell if you did not upload a baseline summary.

In [ ]:
from pathlib import Path
import shutil

baseline_name = "grpo_training_summary.json"
if Path(baseline_name).exists():
    Path("results").mkdir(exist_ok=True)
    shutil.move(baseline_name, "results/grpo_training_summary.json")
    print("baseline summary copied into results/")
else:
    print("No uploaded baseline summary found, comparison plot will use multi-turn only.")

## 7. Verify the multi-turn dataset mix

In [ ]:
from train_grpo_multiturn import build_dataset_records

records = build_dataset_records()
print("dataset size:", len(records))
print("episode examples:", sum(r["record_type"] == "episode" for r in records))
print("trap examples:", sum(r["trajectory_id"] == "trap_low_risk" for r in records))
print("emergency examples:", sum(r["trajectory_id"].startswith("emergency_") for r in records))
print("sample task ids:", [r["task_id"] for r in records[:5]])

## 8. Run multi-turn GRPO training

In [ ]:
!python train_grpo_multiturn.py --use-unsloth --output-dir ./artifacts/niva-grpo-multiturn

## 9. Inspect artifacts

In [ ]:
!ls ./artifacts/niva-grpo-multiturn

## 10. View the comparison reward curve

In [ ]:
from IPython.display import Image, display
from pathlib import Path

plot_path = Path("./artifacts/niva-grpo-multiturn/comparison_reward_curve.png")
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("comparison_reward_curve.png not found")

## 11. View the mixed-signals before/after dialogue

In [ ]:
from pathlib import Path

dialogue_path = Path("./artifacts/niva-grpo-multiturn/mixed_signals_before_after.txt")
if dialogue_path.exists():
    print(dialogue_path.read_text()[:12000])
else:
    print("mixed_signals_before_after.txt not found")

## 12. Zip the trained model folder

In [ ]:
!zip -r niva-grpo-multiturn.zip ./artifacts/niva-grpo-multiturn